<a href="https://colab.research.google.com/github/IvKorolev/Ml_hw/blob/main/Lab4_parallel_comp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Решение

In [ ]:
%%writefile gauss.cu
#include <iostream>
#include <vector>
#include <iomanip>
#include <cmath>
#include <cstdlib>

#include <thrust/extrema.h>
#include <thrust/device_ptr.h>

#define CSC(call)                                                     \
do {                                                                  \
    cudaError_t res = call;                                           \
    if (res != cudaSuccess) {                                         \
        std::cerr << "ERROR in " << __FILE__ << ":" << __LINE__       \
                  << ". Message: " << cudaGetErrorString(res) << "\n";\
        exit(EXIT_FAILURE);                                           \
    }                                                                 \
} while(0)

const double EPS = 1e-7;

struct AbsComparator {
    __host__ __device__
    bool operator()(double a, double b) const {
        return fabs(a) < fabs(b);
    }
};

__global__ void swap_rows_kernel(double *mat, int n, int cols, int r1, int r2) {
    int col = blockIdx.x * blockDim.x + threadIdx.x;
    if (col >= cols) return;

    double tmp = mat[col * n + r1];
    mat[col * n + r1] = mat[col * n + r2];
    mat[col * n + r2] = tmp;
}

__global__ void eliminate_kernel(double *mat, int n, int cols, int k) {

    int row = blockIdx.x * blockDim.x + threadIdx.x + (k + 1);
    int col = blockIdx.y * blockDim.y + threadIdx.y + (k + 1);

    if (row >= n || col >= cols) return;

    double pivot = mat[k * n + k];

    // читаем элемент столбца k в строке row
    double factor = mat[k * n + row] / pivot;

    // обновляем элемент (row, col)
    mat[col * n + row] -= factor * mat[col * n + k];
}

int main() {
    std::ios::sync_with_stdio(false);
    std::cin.tie(nullptr);

    int n;
    if (!(std::cin >> n)) {
        std::cerr << "Cannot read n\n";
        return 1;
    }

    if (n <= 0) {
        std::cerr << "Invalid n\n";
        return 1;
    }

    int cols = n + 1;
    size_t total = (size_t)n * cols;

    // column-major storage for augmented matrix [A|b]
    std::vector<double> host_mat(total);

    // read A
    for (int row = 0; row < n; row++) {
        for (int col = 0; col < n; col++) {
            double val;
            std::cin >> val;
            host_mat[col * n + row] = val;
        }
    }

    // read b as last column
    for (int row = 0; row < n; row++) {
        double val;
        std::cin >> val;
        host_mat[n * n + row] = val; // last column
    }

    double *dev_mat = nullptr;
    CSC(cudaMalloc(&dev_mat, total * sizeof(double)));
    CSC(cudaMemcpy(dev_mat, host_mat.data(), total * sizeof(double), cudaMemcpyHostToDevice));

    AbsComparator comp;

    for (int k = 0; k < n; k++) {
        // find pivot in column k among rows k..n-1
        thrust::device_ptr<double> col_begin = thrust::device_pointer_cast(dev_mat + k * n + k);
        thrust::device_ptr<double> col_end   = thrust::device_pointer_cast(dev_mat + k * n + n);

        thrust::device_ptr<double> pivot_it = thrust::max_element(col_begin, col_end, comp);
        int pivot_row = k + (int)(pivot_it - col_begin);

        double pivot_value;
        CSC(cudaMemcpy(&pivot_value, dev_mat + k * n + pivot_row, sizeof(double), cudaMemcpyDeviceToHost));

        if (fabs(pivot_value) < EPS) {
            std::cerr << "Matrix is singular or system has no unique solution\n";
            CSC(cudaFree(dev_mat));
            return 1;
        }

        // swap rows if needed
        if (pivot_row != k) {
            int blockSize = 256;
            int gridSize = (cols + blockSize - 1) / blockSize;
            swap_rows_kernel<<<gridSize, blockSize>>>(dev_mat, n, cols, k, pivot_row);
            CSC(cudaGetLastError());
            CSC(cudaDeviceSynchronize());
        }

        // eliminate below pivot
        if (k + 1 < n) {
            dim3 block(16, 16);

            dim3 grid(
                (n    - (k + 1) + block.x - 1) / block.x,   // rows
                (cols - (k + 1) + block.y - 1) / block.y    // columns
            );

            eliminate_kernel<<<grid, block>>>(dev_mat, n, cols, k);
            CSC(cudaGetLastError());
            CSC(cudaDeviceSynchronize());
        }
    }

    CSC(cudaMemcpy(host_mat.data(), dev_mat, total * sizeof(double), cudaMemcpyDeviceToHost));
    CSC(cudaFree(dev_mat));

    std::vector<double> x(n, 0.0);

    for (int i = n - 1; i >= 0; i--) {
        double sum = host_mat[n * n + i]; // rhs
        for (int j = i + 1; j < n; j++) {
            sum -= host_mat[j * n + i] * x[j];
        }

        double diag = host_mat[i * n + i];
        if (fabs(diag) < EPS) {
            std::cerr << "Zero diagonal during back substitution\n";
            return 1;
        }

        x[i] = sum / diag;
    }

    std::cout << std::scientific << std::setprecision(10);
    for (int i = 0; i < n; i++) {
        std::cout << x[i];
        if (i + 1 < n) std::cout << " ";
    }
    std::cout << "\n";

    return 0;
}

Overwriting gauss.cu


Вторая попытка

In [101]:
%%writefile gauss.cu
#include <iostream>
#include <vector>
#include <iomanip>
#include <cmath>
#include <cstdlib>

#include <thrust/extrema.h>
#include <thrust/device_ptr.h>
#include <thrust/execution_policy.h>

#define CSC(call)                                                     \
do {                                                                  \
    cudaError_t res = call;                                           \
    if (res != cudaSuccess) {                                         \
        std::cerr << "ERROR in " << __FILE__ << ":" << __LINE__       \
                  << ". Message: " << cudaGetErrorString(res) << "\n";\
        exit(EXIT_FAILURE);                                           \
    }                                                                 \
} while(0)

const double EPS = 1e-7;

// Компаратор по модулю, как требовал преподаватель
struct AbsComparator {
    __host__ __device__
    bool operator()(double a, double b) const {
        return fabs(a) < fabs(b);
    }
};

// ОПТИМИЗАЦИЯ 1: Меняем элементы строк только начиная со столбца k (до k там нули)
__global__ void swap_rows_kernel(double *mat, int n, int cols, int r1, int r2, int start_col) {
    int col = blockIdx.x * blockDim.x + threadIdx.x + start_col;
    if (col < cols) {
        double tmp = mat[col * n + r1];
        mat[col * n + r1] = mat[col * n + r2];
        mat[col * n + r2] = tmp;
    }
}

// ОПТИМИЗАЦИЯ 2: Отдельное ядро для деления O(N^2). Заменяем элемент на его множитель.
__global__ void divide_col_kernel(double *mat, int n, int k) {
    int row = blockIdx.x * blockDim.x + threadIdx.x + k + 1;
    if (row < n) {
        mat[k * n + row] /= mat[k * n + k];
    }
}

// ОПТИМИЗАЦИЯ 3: Деления больше нет! Только быстрое умножение и вычитание (FMA)
__global__ void eliminate_kernel(double *mat, int n, int cols, int k) {
    int row = blockIdx.x * blockDim.x + threadIdx.x + k + 1;
    int col = blockIdx.y * blockDim.y + threadIdx.y + k + 1;

    if (row < n && col < cols) {
        // Берем готовый множитель, который мы вычислили в divide_col_kernel
        double factor = mat[k * n + row];
        double top_val = mat[col * n + k];

        mat[col * n + row] -= factor * top_val;
    }
}

int main() {
    // Ускоряем ввод-вывод
    std::ios::sync_with_stdio(false);
    std::cin.tie(nullptr);

    int n;
    if (!(std::cin >> n) || n <= 0) {
        return 1;
    }

    int cols = n + 1;
    size_t total = (size_t)n * cols;

    std::vector<double> host_mat(total);

    // Читаем матрицу A (храним по столбцам, как требовалось)
    for (int row = 0; row < n; row++) {
        for (int col = 0; col < n; col++) {
            std::cin >> host_mat[col * n + row];
        }
    }

    // Читаем вектор b (как последний столбец)
    for (int row = 0; row < n; row++) {
        std::cin >> host_mat[n * n + row];
    }

    double *dev_mat = nullptr;
    CSC(cudaMalloc(&dev_mat, total * sizeof(double)));
    CSC(cudaMemcpy(dev_mat, host_mat.data(), total * sizeof(double), cudaMemcpyHostToDevice));

    AbsComparator comp;

    // ===============================
    // ПРЯМОЙ ХОД НА GPU
    // ===============================
    for (int k = 0; k < n; k++) {

        // Ищем главный элемент по модулю с помощью Thrust
        thrust::device_ptr<double> col_begin = thrust::device_pointer_cast(dev_mat + k * n + k);
        thrust::device_ptr<double> col_end   = thrust::device_pointer_cast(dev_mat + k * n + n);

        thrust::device_ptr<double> pivot_it = thrust::max_element(thrust::device, col_begin, col_end, comp);
        int pivot_row = k + static_cast<int>(pivot_it - col_begin);

        // Возвращаем значение пивота на хост для проверки на ноль (сингулярность матрицы)
        double pivot_value;
        CSC(cudaMemcpy(&pivot_value, dev_mat + k * n + pivot_row, sizeof(double), cudaMemcpyDeviceToHost));
        if (fabs(pivot_value) < EPS) {
            std::cerr << "Singular matrix.\n";
            CSC(cudaFree(dev_mat));
            return 1;
        }

        // Перестановка строк
        if (pivot_row != k) {
            int swap_cols = cols - k; // Нет смысла свапать нули слева от k
            int blockSize = 256;
            int gridSize = (swap_cols + blockSize - 1) / blockSize;

            swap_rows_kernel<<<gridSize, blockSize>>>(dev_mat, n, cols, k, pivot_row, k);
            CSC(cudaGetLastError());
        }

        if (k + 1 < n) {
            // 1. Вычисляем множители (1D ядро)
            int divide_rows = n - (k + 1);
            int blockSizeDiv = 256;
            int gridSizeDiv = (divide_rows + blockSizeDiv - 1) / blockSizeDiv;
            divide_col_kernel<<<gridSizeDiv, blockSizeDiv>>>(dev_mat, n, k);
            CSC(cudaGetLastError());

            // 2. Обновляем оставшуюся часть матрицы (2D ядро)
            int cols_to_update = cols - (k + 1);

            // Используем блок (32, 8). 32 по X дает идеальное выравнивание под варп (Coalesced access)
            dim3 block(32, 8);
            dim3 grid(
                (divide_rows + block.x - 1) / block.x,
                (cols_to_update + block.y - 1) / block.y
            );

            eliminate_kernel<<<grid, block>>>(dev_mat, n, cols, k);
            CSC(cudaGetLastError());
        }
    }

    // Синхронизируемся в самом конце
    CSC(cudaDeviceSynchronize());
    CSC(cudaMemcpy(host_mat.data(), dev_mat, total * sizeof(double), cudaMemcpyDeviceToHost));
    CSC(cudaFree(dev_mat));

    // ===============================
    // ОБРАТНЫЙ ХОД НА CPU
    // ===============================
    std::vector<double> x(n, 0.0);

    for (int i = n - 1; i >= 0; i--) {
        double sum = host_mat[n * n + i];

        for (int j = i + 1; j < n; j++) {
            sum -= host_mat[j * n + i] * x[j];
        }

        x[i] = sum / host_mat[i * n + i];
    }

    // Вывод с требуемой точностью (setprecision(10) с scientific как раз дает относительную точность 10^-10)
    std::cout << std::scientific << std::setprecision(10);
    for (int i = 0; i < n; i++) {
        std::cout << x[i];
        if (i + 1 < n) std::cout << " ";
    }
    std::cout << "\n";

    return 0;
}

Overwriting gauss.cu


In [102]:
%%shell
nvcc -arch=sm_75 gauss.cu -o gauss

In [103]:
%%shell
printf "2\n1 2\n3 4\n5 6\n" | ./gauss

-4.0000000000e+00 4.5000000000e+00


In [104]:
%%shell
printf "3\n1 2 3\n4 5 6\n7 8 7\n1 2 3\n" | ./gauss

-3.3333333333e-01 6.6666666667e-01 -1.5860328923e-17


In [105]:
%%shell
printf "3\n5 0 0\n0 3 0\n0 0 2\n10 9 8\n" | ./gauss

2.0000000000e+00 3.0000000000e+00 4.0000000000e+00


In [106]:
%%shell
printf "2\n0 2\n3 1\n4 5\n" | ./gauss

1.0000000000e+00 2.0000000000e+00


In [107]:
%%shell
printf "2\n-10 2\n3 4\n-8 10\n" | ./gauss

1.1304347826e+00 1.6521739130e+00


In [108]:
%%shell
printf "4\n\
2 1 -1 3\n\
4 5 -3 6\n\
-2 5 -2 6\n\
4 11 -4 8\n\
5 9 4 2\n" | ./gauss

-1.8181818182e-01 -2.3636363636e+00 -6.0909090909e+00 5.4545454545e-01


In [109]:
%%shell
./gauss < test.txt

3.1906080148e-01 -8.0553910934e-01 2.9941463678e+00 -1.7592881474e+00 6.7093781606e-01 -7.4762311264e-01 4.6684611183e-01 2.9079104113e+00 9.7173459175e-01 1.4767833103e+00 -2.1052492659e+00 7.0815257038e-01 7.6790164487e-01 -1.1734573354e+00 9.3147308074e-01 -1.7542588860e+00 4.8292074170e-01 -2.8078795248e-01 4.9522929054e-01 -6.0633554841e-01 8.9989575803e-02 -1.6814141312e-02 -5.3023552816e-01 3.9805599639e-01 4.1348571814e-01 7.0607163265e-02 -3.6457174540e-01 1.3757420449e+00 -1.0647328191e+00 -2.2220350658e-01 -2.7505056947e-01 6.1303244912e-01 2.0662966612e+00 -5.0546388886e-01 2.1116807765e-01 -6.6490623707e-01 -5.4284517272e-01 -4.0939389804e-01 8.8784066397e-01 -8.0173397426e-01 4.3593550018e-01 5.4028874527e-01 -8.0623570087e-01 9.3014069281e-01 -2.3583111238e+00 -8.0941336426e-01 4.9109805203e-02 8.9500093606e-01 -2.7946570712e-01 4.4576223109e-01 2.3248171773e+00 1.0758457255e+00 -9.3022932841e-01 -5.9853390594e-01 -2.5089547057e+00 7.7289989962e-01 4.8399171566e-01 8.182

In [ ]:
import random

def generate_test(n, filename):
    with open(filename, "w") as f:
        f.write(f"{n}\n")
        for i in range(n):
            row = [random.uniform(-10,10) for _ in range(n)]
            f.write(" ".join(map(str,row)) + "\n")
        for i in range(n):
            f.write(str(random.uniform(-10,10)) + "\n")

generate_test(1500, "big_test.txt")

In [ ]:
%%shell
./gauss < big_test.txt > result.txt

# Замеры времени

In [6]:
%%writefile gauss.cu
#include <iostream>
#include <vector>
#include <iomanip>
#include <cmath>
#include <cstdlib>
#include <chrono>

#include <thrust/extrema.h>
#include <thrust/device_ptr.h>
#include <thrust/execution_policy.h>

#define CSC(call)                                                     \
do {                                                                  \
    cudaError_t res = call;                                           \
    if (res != cudaSuccess) {                                         \
        std::cerr << "ERROR in " << __FILE__ << ":" << __LINE__       \
                  << ". Message: " << cudaGetErrorString(res) << "\n";\
        exit(EXIT_FAILURE);                                           \
    }                                                                 \
} while(0)

const double EPS = 1e-7;

struct AbsComparator {
    __host__ __device__
    bool operator()(double a, double b) const {
        return fabs(a) < fabs(b);
    }
};

// 1. Ядро для перестановки строк (начинаем со start_col, чтобы не двигать уже полученные нули)
__global__ void swap_rows_kernel(double *mat, int n, int cols, int r1, int r2, int start_col) {
    int col = blockIdx.x * blockDim.x + threadIdx.x + start_col;
    if (col < cols) {
        double tmp = mat[col * n + r1];
        mat[col * n + r1] = mat[col * n + r2];
        mat[col * n + r2] = tmp;
    }
}

// 2. Ядро для вычисления множителей (одномерная сетка) - избавляет нас от деления в 2D ядре
__global__ void divide_col_kernel(double *mat, int n, int k) {
    int row = blockIdx.x * blockDim.x + threadIdx.x + k + 1;
    if (row < n) {
        mat[k * n + row] /= mat[k * n + k];
    }
}

// 3. Основное ядро - только быстрое умножение и вычитание
__global__ void eliminate_kernel(double *mat, int n, int cols, int k) {
    int row = blockIdx.x * blockDim.x + threadIdx.x + k + 1;
    int col = blockIdx.y * blockDim.y + threadIdx.y + k + 1;

    if (row < n && col < cols) {
        double factor = mat[k * n + row];
        double top_val = mat[col * n + k];
        mat[col * n + row] -= factor * top_val;
    }
}

// ==========================================
// CPU версия прямого хода для замеров времени
// Передаем по значению (копия), чтобы не испортить исходную матрицу
// ==========================================
void gauss_cpu(std::vector<double> mat, int n) {
    int cols = n + 1;

    for (int k = 0; k < n; k++) {
        // Поиск максимального по модулю
        int pivot = k;
        for (int i = k + 1; i < n; i++) {
            if (fabs(mat[k * n + i]) > fabs(mat[k * n + pivot])) {
                pivot = i;
            }
        }
        if (fabs(mat[k * n + pivot]) < EPS) return; // Сингулярная

        // Перестановка
        if (pivot != k) {
            for (int col = k; col < cols; col++) {
                std::swap(mat[col * n + k], mat[col * n + pivot]);
            }
        }

        // Исключение
        for (int i = k + 1; i < n; i++) {
            double factor = mat[k * n + i] / mat[k * n + k];
            for (int col = k + 1; col < cols; col++) {
                mat[col * n + i] -= factor * mat[col * n + k];
            }
        }
    }
}

int main(int argc, char** argv) {
    std::ios::sync_with_stdio(false);
    std::cin.tie(nullptr);

    // Считываем размеры блока из аргументов (если есть), иначе 32x8
    int block_x = 32;
    int block_y = 8;
    if (argc >= 3) {
        block_x = std::atoi(argv[1]);
        block_y = std::atoi(argv[2]);
    }

    int n;
    if (!(std::cin >> n) || n <= 0) return 1;

    int cols = n + 1;
    size_t total = (size_t)n * cols;
    std::vector<double> host_mat(total);

    for (int row = 0; row < n; row++)
        for (int col = 0; col < n; col++)
            std::cin >> host_mat[col * n + row];

    for (int row = 0; row < n; row++)
        std::cin >> host_mat[n * n + row];

    // Выводим инфу в stderr
    std::cerr << "Matrix size N: " << n << "\n";
    std::cerr << "Block size : " << block_x << " x " << block_y << "\n";

    // --- Замер CPU ---
    auto cpu_start = std::chrono::high_resolution_clock::now();
    gauss_cpu(host_mat, n); // Вызываем от копии
    auto cpu_end = std::chrono::high_resolution_clock::now();
    double cpu_ms = std::chrono::duration<double, std::milli>(cpu_end - cpu_start).count();

    // --- Подготовка GPU ---
    double *dev_mat = nullptr;
    CSC(cudaMalloc(&dev_mat, total * sizeof(double)));
    CSC(cudaMemcpy(dev_mat, host_mat.data(), total * sizeof(double), cudaMemcpyHostToDevice));

    AbsComparator comp;

    cudaEvent_t start_gpu, stop_gpu;
    CSC(cudaEventCreate(&start_gpu));
    CSC(cudaEventCreate(&stop_gpu));

    // --- Замер GPU ---
    CSC(cudaEventRecord(start_gpu));

    for (int k = 0; k < n; k++) {
        thrust::device_ptr<double> col_begin = thrust::device_pointer_cast(dev_mat + k * n + k);
        thrust::device_ptr<double> col_end   = thrust::device_pointer_cast(dev_mat + k * n + n);

        thrust::device_ptr<double> pivot_it = thrust::max_element(thrust::device, col_begin, col_end, comp);
        int pivot_row = k + static_cast<int>(pivot_it - col_begin);

        double pivot_value;
        CSC(cudaMemcpy(&pivot_value, dev_mat + k * n + pivot_row, sizeof(double), cudaMemcpyDeviceToHost));

        if (fabs(pivot_value) < EPS) {
            std::cerr << "Singular matrix.\n";
            CSC(cudaFree(dev_mat));
            return 1;
        }

        if (pivot_row != k) {
            int swap_cols = cols - k;
            int bs = 256; // Для 1D ядра перестановки можно оставить 256
            int gs = (swap_cols + bs - 1) / bs;
            swap_rows_kernel<<<gs, bs>>>(dev_mat, n, cols, k, pivot_row, k);
            CSC(cudaGetLastError());
        }

        if (k + 1 < n) {
            // Деление (1D ядро)
            int divide_rows = n - (k + 1);
            int bs_div = 256;
            int gs_div = (divide_rows + bs_div - 1) / bs_div;
            divide_col_kernel<<<gs_div, bs_div>>>(dev_mat, n, k);
            CSC(cudaGetLastError());

            // Исключение (2D ядро с настраиваемыми размерами блока)
            int cols_to_update = cols - (k + 1);
            dim3 block(block_x, block_y);
            dim3 grid(
                (divide_rows + block.x - 1) / block.x,
                (cols_to_update + block.y - 1) / block.y
            );
            eliminate_kernel<<<grid, block>>>(dev_mat, n, cols, k);
            CSC(cudaGetLastError());
        }
    }

    CSC(cudaEventRecord(stop_gpu));
    CSC(cudaEventSynchronize(stop_gpu));

    float gpu_ms = 0;
    CSC(cudaEventElapsedTime(&gpu_ms, start_gpu, stop_gpu));

    CSC(cudaMemcpy(host_mat.data(), dev_mat, total * sizeof(double), cudaMemcpyDeviceToHost));
    CSC(cudaFree(dev_mat));
    CSC(cudaEventDestroy(start_gpu));
    CSC(cudaEventDestroy(stop_gpu));

    std::cerr << "CPU time: " << cpu_ms << " ms\n";
    std::cerr << "GPU forward time: " << gpu_ms << " ms\n";
    std::cerr << "Speedup: " << cpu_ms / gpu_ms << "x\n\n";

    // --- Обратный ход (всегда на CPU) ---
    std::vector<double> x(n, 0.0);
    for (int i = n - 1; i >= 0; i--) {
        double sum = host_mat[n * n + i];
        for (int j = i + 1; j < n; j++) {
            sum -= host_mat[j * n + i] * x[j];
        }
        x[i] = sum / host_mat[i * n + i];
    }

    std::cout << std::scientific << std::setprecision(10);
    for (int i = 0; i < n; i++) {
        std::cout << x[i];
        if (i + 1 < n) std::cout << " ";
    }
    std::cout << "\n";

    return 0;
}

Overwriting gauss.cu


In [7]:
%%shell
nvcc -arch=sm_75 gauss.cu -o gauss

/bin/bash: line 1: nvcc: command not found


CalledProcessError: Command 'nvcc -arch=sm_75 gauss.cu -o gauss
' returned non-zero exit status 127.

In [112]:
%%shell
echo "--- Тестирование сетки 32 x 8 ---"
./gauss 32 8 < test.txt > result1.txt
md5sum result1.txt

--- Тестирование сетки 32 x 8 ---
Matrix size N: 1000
Block size : 32 x 8
CPU time: 3796.46 ms
GPU forward time: 172.48 ms
Speedup: 22.011x

95b2715d93c7684f2bc718b715c2db36  result1.txt


In [113]:
%%shell
echo "--- Тестирование сетки 16 x 16 ---"
./gauss 16 16 < test.txt > result2.txt
md5sum result2.txt

--- Тестирование сетки 16 x 16 ---
Matrix size N: 1000
Block size : 16 x 16
CPU time: 3527.94 ms
GPU forward time: 169.091 ms
Speedup: 20.8642x

95b2715d93c7684f2bc718b715c2db36  result2.txt


In [114]:
%%shell
echo "--- Тестирование сетки 8 x 32 ---"
./gauss 8 32 < test.txt > result3.txt
md5sum result3.txt

--- Тестирование сетки 8 x 32 ---
Matrix size N: 1000
Block size : 8 x 32
CPU time: 4113.49 ms
GPU forward time: 220.241 ms
Speedup: 18.6772x

95b2715d93c7684f2bc718b715c2db36  result3.txt


In [115]:
%%shell
diff result1.txt result2.txt
diff result1.txt result3.txt

In [1]:
import random

def generate_test(n, filename):
    with open(filename, "w") as f:
        f.write(f"{n}\n")

        # Матрица A
        for i in range(n):
            row = [random.uniform(-10, 10) for _ in range(n)]
            f.write(" ".join(map(str, row)) + "\n")

        # Вектор b
        for i in range(n):
            f.write(str(random.uniform(-10, 10)) + "\n")

# Размер можно менять
generate_test(500, "test.txt")

print("Test file generated: test.txt")

Test file generated: test.txt


In [ ]:
%%shell
nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
%%shell
which nvprof

/usr/local/cuda/bin/nvprof


In [124]:
%%shell
nvprof ./gauss < test.txt > result.txt

Matrix size N: 1000
Block size : 32 x 8
==60912== NVPROF is profiling process 60912, command: ./gauss
CPU time: 6527.51 ms
GPU forward time: 287.996 ms
Speedup: 22.6653x

==60912== Profiling application: ./gauss
==60912== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   54.03%  23.275ms       999  23.298us  2.1440us  70.238us  eliminate_kernel(double*, int, int, int)
                   14.47%  6.2350ms      1000  6.2350us  2.9760us  8.5120us  _ZN6thrust20THRUST_200700_750_NS8cuda_cub4core13_kernel_agentINS1_8__reduce11ReduceAgentINS0_12zip_iteratorINS0_5tupleIJNS0_10device_ptrIdEENS1_19counting_iterator_tIlEEEEEEEPNS7_IJdlEEESE_iNS1_9__extrema9arg_max_fIdl13AbsComparatorEEEEJSD_SF_iSJ_EEEvDpT0_
                   12.05%  5.1926ms      2001  2.5950us  1.5990us  1.5835ms  [CUDA memcpy DtoH]
                    8.38%  3.6112ms       991  3.6430us  2.0160us  4.6720us  swap_rows_kernel(double*, int, int, int, int, int)
  

In [122]:
%%shell
nvprof -e divergent_branch,global_store_transaction,l1_local_load_hit,l1_shared_bank_conflict -m sm_efficiency ./gauss < test.txt

======== Warning: Skipping profiling on device 0 since profiling is not supported on devices with compute capability 7.5 and higher.
                  Use NVIDIA Nsight Compute for GPU profiling and NVIDIA Nsight Systems for GPU tracing and CPU sampling.
                  Refer https://developer.nvidia.com/tools-overview for more details.

Matrix size N: 1000
Block size : 32 x 8
==49329== NVPROF is profiling process 49329, command: ./gauss
CPU time: 3563.19 ms
GPU forward time: 185.479 ms
Speedup: 19.2107x

3.1906080148e-01 -8.0553910934e-01 2.9941463678e+00 -1.7592881474e+00 6.7093781606e-01 -7.4762311264e-01 4.6684611183e-01 2.9079104113e+00 9.7173459175e-01 1.4767833103e+00 -2.1052492659e+00 7.0815257038e-01 7.6790164487e-01 -1.1734573354e+00 9.3147308074e-01 -1.7542588860e+00 4.8292074170e-01 -2.8078795248e-01 4.9522929054e-01 -6.0633554841e-01 8.9989575803e-02 -1.6814141312e-02 -5.3023552816e-01 3.9805599639e-01 4.1348571814e-01 7.0607163265e-02 -3.6457174540e-01 1.3757420449e+00 